# Cross-soup v3×v2 + Symmetry TTA

Публичный notebook: **Stage-A** → **v2 Stage-B** + **v3 Stage-B** → cross-soup grid → submit **с TTA**.

Формула: **`(1-α)·v2_step2000 + α·v3_step400`**. Grid search по α, eval **с symmetry TTA** (как в submit).

- **pair_text v1** обязателен
- inference: `score = 0.5·p(a,b) + 0.5·p(b,a)`
- reference offline: gray **~0.555**, best LB cross-soup **0.5522** + TTA delta **~0.001**

In [1]:
import json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
import torch, transformers

START = Path.cwd().resolve()
masked = lambda path: f"***/{Path(path).name}"

def resolve_script(env_name: str, default: str) -> Path:
    rel = os.getenv(env_name, default)
    p = Path(rel).expanduser()
    if p.is_absolute() and p.exists():
        return p.resolve()
    for base in [START, *START.parents]:
        cand = (base / rel).resolve()
        if cand.exists():
            return cand
    raise FileNotFoundError(f"Не найден {rel}")

def find_file(name: str, *roots: Path) -> Path:
    for root in roots:
        p = (root / name).resolve()
        if p.exists():
            return p
    raise FileNotFoundError(name)

TRAIN_V3 = resolve_script("STAGE_B_V3_SCRIPT", "train_bge_stageb_v3.py")
BLEND_SCRIPT = resolve_script("STAGE_B_BLEND_SCRIPT", "final_4_models/scripts/blend_checkpoint_soup.py")
BUILD_SUBMIT = resolve_script("TTA_BUILD_SUBMIT", "notebooks/symmetry_tta_v3_soup/build_submit.py")
SUBMIT_TEMPLATE = resolve_script("TTA_SUBMIT_TEMPLATE", "final_4_models/submit/matching-bge-human-ft/src/utils.py")

PROJECT_DIR = TRAIN_V3.parent
DATA_DIR = Path(os.getenv("BGE_DATA_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_ROOT = Path(os.getenv("TTA_OUTPUT_ROOT", START / "output")).expanduser().resolve()
LIB = find_file("pair_text_v1.py", START / "../lib", START.parent / "lib", PROJECT_DIR / "notebooks/lib")
sys.path.insert(0, str(LIB.parent))

STAGE_A_RUN = os.getenv("STAGE_A_RUN_ID", "research_worker_s42_02")
STAGE_A_OUT = Path(os.getenv(
    "STAGE_A_OUTPUT",
    START.parent / "initial_stage_a_user_bge" / "output" / STAGE_A_RUN,
)).expanduser().resolve()
STAGE_A_INIT = Path(os.getenv("STAGE_A_INIT", STAGE_A_OUT / "best.pt")).expanduser().resolve()

V2_RUN = Path(os.getenv(
    "V2_STAGE_B_OUT",
    START.parent / "v2_soup_lb_5522" / "output" / "v2_soup_s42_01" / "stageb_v2",
)).expanduser().resolve()
V2_CKPT_B = Path(os.getenv("V2_CKPT_2000", V2_RUN / "checkpoints/step_02000.pt")).resolve()

assert TRAIN_V3.exists() and BLEND_SCRIPT.exists() and BUILD_SUBMIT.exists()
assert STAGE_A_INIT.exists(), f"Нет Stage-A init: ***/{STAGE_A_INIT.name}"
assert V2_CKPT_B.exists(), f"Нет v2 step_2000 — сначала v2_soup pipeline"
assert "symmetry TTA" in SUBMIT_TEMPLATE.read_text() or "enc_rev" in SUBMIT_TEMPLATE.read_text()

EXPECTED_VERSIONS = {
    "python": "3.12", "torch": "2.6.0+cu124", "transformers": "4.57.6",
    "numpy": "2.2.6", "pandas": "2.3.3", "pyarrow": "23.0.1", "scikit-learn": "1.8.0",
}
ACTUAL = {
    "python": ".".join(map(str, sys.version_info[:2])),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__, "scikit-learn": sklearn.__version__,
}
assert ACTUAL == EXPECTED_VERSIONS, ACTUAL

print("v3 train:", masked(TRAIN_V3))
print("blend:", masked(BLEND_SCRIPT))
print("stage-a init:", masked(STAGE_A_INIT), "OK")
print("v2 step_2000:", masked(V2_CKPT_B.name), "OK")
print("submit template: symmetry TTA OK")

v3 train: ***/train_bge_stageb_v3.py
blend: ***/blend_checkpoint_soup.py
stage-a init: ***/best.pt OK
v2 step_2000: ***/step_02000.pt OK
submit template: symmetry TTA OK


In [2]:
from verify_pair_text import compare_v1_vs_v2, main as verify_main
assert verify_main() == 0
cmp = compare_v1_vs_v2()
print(json.dumps({k: v for k, v in cmp.items() if not k.endswith("_preview")}, ensure_ascii=False, indent=2))

pair_text v1 verification
  version=v1 attr_limit=520
  v1 vs v2 equal: False (must be False for fashion stress)
  v2 size-first: True | v1 size-first: False
OK — use pair_text_v1 for v2 soup & symmetry TTA pipelines
{
  "v1_len": 181,
  "v2_len": 181,
  "v1_has_size_first": false,
  "v2_has_size_first": true,
  "texts_equal": false
}


In [3]:
RUN_ID = "cross_tta_s42_01"
GPU_IDS = "0,1"
NPROC = len(GPU_IDS.split(","))
MAX_USED_MIB = 15_000
MIN_FREE_MIB = 55_000

RUN_V3 = True          # v3 Stage-B от нового Stage-A init
RUN_BLEND = True       # cross-soup grid + TTA eval
RUN_SUBMIT = True
SYMMETRY_TTA_EVAL = True   # grid search с TTA (как submit)

# fine grid вокруг best α=0.10 (cross-soup reference)
ALPHA_GRID = "0.05,0.08,0.10,0.12,0.15,0.18,0.20,0.25,0.30"

V3_OUT = OUTPUT_ROOT / RUN_ID / "stageb_v3"
SOUP_OUT = OUTPUT_ROOT / RUN_ID / "soup_run"
LOG_V3 = OUTPUT_ROOT / RUN_ID / "stageb_v3.log"
V3_OUT.mkdir(parents=True, exist_ok=True)
SOUP_OUT.mkdir(parents=True, exist_ok=True)

CKPT_A = V3_OUT / "checkpoints/step_00400.pt"   # v3 high-gray
CKPT_B = V2_CKPT_B                               # v2 high-problem

CONFIG = {
    "stage_a_init": f"***/{STAGE_A_INIT.name}",
    "v2_ckpt_2000": f"***/{CKPT_B.name}",
    "v3_out": f"***/{V3_OUT.name}",
    "soup_out": f"***/{SOUP_OUT.name}",
    "blend_formula": "(1-alpha)*v2_step2000 + alpha*v3_step400",
    "alpha_grid": ALPHA_GRID,
    "symmetry_tta_eval": SYMMETRY_TTA_EVAL,
    "physical_gpu_ids": GPU_IDS,
    "run_v3": RUN_V3,
    "run_blend": RUN_BLEND,
    "run_submit": RUN_SUBMIT,
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

def run_cmd(cmd: str, env: dict | None = None, log_path: Path | None = None) -> int:
    env = {**os.environ, **(env or {})}
    shown = cmd
    for secret in (str(PROJECT_DIR), str(DATA_DIR), str(V3_OUT), str(SOUP_OUT), str(STAGE_A_INIT), str(V2_CKPT_B)):
        shown = shown.replace(secret, "***")
    print("$", shown, flush=True)
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("a", encoding="utf-8") as f:
            f.write(f"$ {shown}\n")
            p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
            assert p.stdout is not None
            for line in p.stdout:
                print(line, end="", flush=True)
                f.write(line)
            return p.wait()
    return subprocess.call(cmd, shell=True, env=env)

def gpu_mem_mib(ids: list[int]) -> tuple[dict[int, int], dict[int, int]]:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"], text=True
    )
    used, total = {}, {}
    for ln in out.strip().splitlines():
        i, u, t = (int(x.strip()) for x in ln.split(","))
        used[i], total[i] = u, t
    sel = [i for i in ids]
    return {i: used[i] for i in sel}, {i: total[i] - used[i] for i in sel}

selected = [int(x) for x in GPU_IDS.split(",")]
used, free = gpu_mem_mib(selected)
print("GPUs:", selected, "used MiB:", used, "free MiB:", free)
assert all(free[i] >= MIN_FREE_MIB for i in selected), free

{
  "stage_a_init": "***/best.pt",
  "v2_ckpt_2000": "***/step_02000.pt",
  "v3_out": "***/stageb_v3",
  "soup_out": "***/soup_run",
  "blend_formula": "(1-alpha)*v2_step2000 + alpha*v3_step400",
  "alpha_grid": "0.05,0.08,0.10,0.12,0.15,0.18,0.20,0.25,0.30",
  "symmetry_tta_eval": true,
  "physical_gpu_ids": "0,1",
  "run_v3": true,
  "run_blend": true,
  "run_submit": true
}
GPUs: [0, 1] used MiB: {0: 1907, 1: 5011} free MiB: {0: 79652, 1: 76548}


In [4]:
STAGE_B_ENV = dict(
    STAGE_B_INIT_CKPT=str(STAGE_A_INIT),
    STAGE_B_OUT_DIR=str(V3_OUT),
    STAGE_B_ITEMS_PATH=str(DATA_DIR / "items.parquet"),
    STAGE_B_ITEMS_HUMAN_PATH=str(DATA_DIR / "items_human.parquet"),
    STAGE_B_MATCHES_HUMAN=str(DATA_DIR / "matches.parquet"),
    STAGE_B_MATCHES_LLM=str(DATA_DIR / "matches_llm.parquet"),
)

if RUN_V3:
    if not (V3_OUT / "teacher_probs.npz").exists():
        rc = run_cmd(
            f"cd {PROJECT_DIR} && CUDA_VISIBLE_DEVICES={GPU_IDS.split(',')[0]} "
            f"python {TRAIN_V3} --precompute-teacher",
            env=STAGE_B_ENV, log_path=LOG_V3,
        )
        assert rc == 0
    else:
        print("teacher cache exists — skip precompute")
    rc = run_cmd(
        f"cd {PROJECT_DIR} && CUDA_VISIBLE_DEVICES={GPU_IDS} "
        f"python -m torch.distributed.run --nproc_per_node={NPROC} {TRAIN_V3}",
        env=STAGE_B_ENV, log_path=LOG_V3,
    )
    assert rc == 0
else:
    print("RUN_V3=False — нужен готовый v3 step_00400")

assert CKPT_A.exists(), "Нет v3 step_00400 для cross-soup"

$ cd *** && CUDA_VISIBLE_DEVICES=0 python ***/train_bge_stageb_v3.py --precompute-teacher
PRECOMPUTE teacher cache (single GPU)
manual split: {'train': 292508, 'tune': 36584, 'eval': 36562}
loading 805,302 texts...
items_human hit 711,304/805,302
items.parquet scan; missing left 0; total texts 805,302
oversample Обувь: 292,508 -> 307,197 human train rows
gray: full=45,831 sample=11,851
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
scoring teacher probs for human train (307,197 pairs)...
  human: 128/307,197 (0.0%) 315 pair/s
  human: 25,728/307,197 (8.4%) 577 pair/s
  human: 51,328/307,197 (16.7%) 579 pair/s
  human: 76,928/307,197 (25.0%) 579 pair/s
  human: 102,528/

In [5]:
if RUN_BLEND:
    scripts_dir = BLEND_SCRIPT.parent
    sys.path.insert(0, str(scripts_dir))
    sys.path.insert(0, str(LIB.parent))
    from verify_pair_text import patch_score_ensemble_v1
    patch_score_ensemble_v1()
    tta_flag = "--symmetry-tta" if SYMMETRY_TTA_EVAL else ""
    rc = run_cmd(
        f"cd {PROJECT_DIR} && PYTHONPATH={scripts_dir}:{LIB.parent} "
        f"python {BLEND_SCRIPT} --ckpt-a {CKPT_A} --ckpt-b {CKPT_B} "
        f"--out-dir {SOUP_OUT} --alphas {ALPHA_GRID} --skip-submit --gpu {GPU_IDS.split(',')[0]} {tta_flag}",
        env=STAGE_B_ENV,
    )
    assert rc == 0

metrics = json.loads((SOUP_OUT / "metrics.json").read_text())
grid = pd.DataFrame(json.loads((SOUP_OUT / "soup_grid.json").read_text()))
display(grid.round(4))
print("best alpha:", metrics["blend"]["best_alpha"])
print("symmetry_tta_eval:", metrics.get("symmetry_tta_eval"))
print(json.dumps(metrics["best_metrics"], indent=2))

REF = {"alpha": 0.10, "gray_full": 0.5547, "problem_ap": 0.6371, "composite": None}
best = metrics["best_metrics"]
print("\nvs reference 04_v3_soup_tta (α=0.10):")
print(f"  gray:   {best['gray_full']:.4f} vs {REF['gray_full']:.4f}")
print(f"  problem:{best['problem_ap']:.4f} vs {REF['problem_ap']:.4f}")

$ cd *** && PYTHONPATH=***/final_4_models/scripts:***/notebooks/lib python ***/final_4_models/scripts/blend_checkpoint_soup.py --ckpt-a ***/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/stageb_v3/checkpoints/step_00400.pt --ckpt-b ***/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_02000.pt --out-dir ***/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/soup_run --alphas 0.05,0.08,0.10,0.12,0.15,0.18,0.20,0.25,0.30 --skip-submit --gpu 0 --symmetry-tta
checkpoint soup | A=/home/dgbabenko/assistant-peft/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/stageb_v3/checkpoints/step_00400.pt | B=/home/dgbabenko/assistant-peft/notebooks/v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_02000.pt
blend: theta = (1-alpha)*B + alpha*A  | alphas=[0.05, 0.08, 0.1, 0.12, 0.15, 0.18, 0.2, 0.25, 0.3]
symmetry_tta=True
OUT=/home/dgbabenko/assistant-peft/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/soup_run
loading 141,261 texts...
 

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  alpha=0.05 (B=0.95): gray=0.5502 problem=0.6409 tune=0.8013 score=0.6458  [298.4s]


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  alpha=0.08 (B=0.92): gray=0.5512 problem=0.6392 tune=0.8007 score=0.6451  [297.9s]


...

  alpha=0.30 (B=0.70): gray=0.5617 problem=0.6257 tune=0.7952 score=0.6404  [301.6s]

BEST alpha=0.05 score=0.6458


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at deepvk/USER-bge-m3 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


saved /home/dgbabenko/assistant-peft/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/soup_run/best.pt /home/dgbabenko/assistant-peft/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/soup_run/export_fp16/ metrics.json


,alpha_step400,weight_step2000,gray_full,problem_ap,tune_macro,composite_score,elapsed_s
0,0.05,0.95,0.5502,0.6409,0.8013,0.6458,298.4
1,0.08,0.92,0.5512,0.6392,0.8007,0.6451,297.9
2,0.10,0.90,0.5522,0.6385,0.8004,0.6450,297.9
3,0.12,0.88,0.5532,0.6374,0.7999,0.6447,298.8
4,0.15,0.85,0.5544,0.6357,0.7992,0.6440,298.2
5,0.18,0.82,0.5559,0.6338,0.7985,0.6434,298.5
6,0.20,0.80,0.5568,0.6330,0.7981,0.6431,297.9
7,0.25,0.75,0.5593,0.6296,0.7967,0.6419,298.0
8,0.30,0.70,0.5617,0.6257,0.7952,0.6404,301.6


best alpha: 0.05
symmetry_tta_eval: True
{
  "alpha_step400": 0.05,
  "weight_step2000": 0.95,
  "gray_full": 0.5501535514366667,
  "problem_ap": 0.6409161421346352,
  "tune_macro": 0.8013128128986512,
  "composite_score": 0.6457666990780478,
  "elapsed_s": 298.4
}

vs reference 04_v3_soup_tta (α=0.10):
  gray:   0.5502 vs 0.5547
  problem:0.6409 vs 0.6371


In [6]:
if RUN_SUBMIT:
    assert run_cmd(f"python {BUILD_SUBMIT} --run-dir {SOUP_OUT}") == 0
    zip_path = SOUP_OUT / "matching-bge-human-ft-submit.zip"
    print("submit:", masked(zip_path), f"{zip_path.stat().st_size/1024**2:.1f} MB")
    print("symmetry TTA: enabled in submit template")
else:
    print("RUN_SUBMIT=False")

$ python ***/notebooks/symmetry_tta_v3_soup/build_submit.py --run-dir ***/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/soup_run
created /home/dgbabenko/assistant-peft/notebooks/symmetry_tta_v3_soup/output/cross_tta_s42_01/soup_run/matching-bge-human-ft-submit.zip (1269.0 MB)
sha256: 5b093cc15550fbb9f3d283da92b78d0c6f6add2a4ea7bd8515f8a027685bf896
submit: ***/matching-bge-human-ft-submit.zip 1269.0 MB
symmetry TTA: enabled in submit template


In [ ]:
import zipfile, tempfile
zip_path = SOUP_OUT / "matching-bge-human-ft-submit.zip"
assert zip_path.exists()
with tempfile.TemporaryDirectory() as td:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(td)
    root = next(Path(td).glob("matching-bge-human-ft*"))
    sys.path.insert(0, str(root))
    os.chdir(root)
    from src.utils import build_text, _score_pairs
    print("import OK:", build_text("test", '{"Бренд":"X"}')[:40])
    print("TTA fn:", _score_pairs.__name__)
print("Готово: cross-soup + symmetry TTA submit")